In [1]:
# Create financial ratios with safe division.
# Return missing values for zero denominators; preserve negative, economically valid results.

In [2]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths, safe_divide
from project_config import RATIO_FORMULAS as ratio_formulas
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/original_features.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
original_features = pd.read_csv(PROCESSED / "original_features.csv")
required_fields = sorted({field for pair in ratio_formulas.values() for field in pair})
missing_fields = sorted(
    set(required_fields + ["event_id", "company_id", "prediction_date", "target"])
    - set(original_features.columns)
)
if missing_fields:
    raise ValueError(f"Missing required original columns: {missing_fields}")
if original_features.empty:
    raise ValueError("No original events; inspect notebook 08")
if original_features[["event_id", "company_id", "prediction_date"]].isna().any().any():
    raise ValueError("Missing company/event identifiers or prediction dates")
if (
    original_features.duplicated(["company_id", "prediction_date"]).any()
    or not original_features["event_id"].is_unique
):
    raise ValueError("Original data must contain one row per company-event")
if set(ratio_formulas) & set(original_features.columns):
    raise ValueError("Original data already contains derived ratio names")
for field in required_fields:
    if not pd.api.types.is_numeric_dtype(original_features[field]):
        raise ValueError(f"Expected a numeric original field: {field}")
    if not np.isfinite(
        original_features.loc[original_features[field].notna(), field]
    ).all():
        raise ValueError(
            f"Nonfinite original values in {field}; inspect upstream cleaning"
        )

In [3]:
# Mask zero denominators before division, and discard nonfinite results.

ratio_values = pd.DataFrame(
    {
        name: safe_divide(original_features[numerator], original_features[denominator])
        for name, (numerator, denominator) in ratio_formulas.items()
    },
    index=original_features.index,
)
derived_features = pd.concat([original_features.copy(), ratio_values], axis=1)
ratio_summary = pd.DataFrame(
    [
        {
            "ratio": name,
            "numerator": numerator,
            "denominator": denominator,
            "missing_ratios": int(derived_features[name].isna().sum()),
            "missing_pct": 100 * derived_features[name].isna().mean(),
            "zero_denominators": int(original_features[denominator].eq(0).sum()),
            "missing_denominators": int(original_features[denominator].isna().sum()),
            "negative_ratios": int(derived_features[name].lt(0).sum()),
        }
        for name, (numerator, denominator) in ratio_formulas.items()
    ]
)

In [4]:
# Verify unchanged original data, row identity, and finite-or-missing ratios.
for name, (numerator, denominator) in ratio_formulas.items():
    value = derived_features[name]
    unusable = (
        original_features[numerator].isna()
        | original_features[denominator].isna()
        | original_features[denominator].eq(0)
    )
    valid_zero = original_features[numerator].eq(0) & ~unusable
    nonzero_result = value.notna() & original_features[numerator].ne(0)
    expected_sign = np.sign(original_features[numerator]) * np.sign(
        original_features[denominator]
    )

output_path = PROCESSED / "derived_features.csv"
derived_features.to_csv(output_path, index=False)